# Contextual Data Fusion and Feature Engineering

## Objective

This notebook prepares machine telemetry signals for contextual feature engineering.

Tasks:

- Load telemetry dataset
- Identify sensor signals
- Create feature engineering workflow
- Prepare for rolling statistics generation
- Build the foundation for contextual data fusion

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
df = pd.read_csv("../data/raw/ai4i2020.csv")

print(df.shape)

df.head()

(10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
sensor_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

sensor_cols

['Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]']

In [5]:
sensor_summary = df[sensor_cols].describe().T

sensor_summary

,count,mean,std,min,25%,50%,75%,max
Air temperature [K],10000.0,300.00493,2.000259,295.3,298.3,300.1,301.5,304.5
Process temperature [K],10000.0,310.00556,1.483734,305.7,308.8,310.1,311.1,313.8
Rotational speed [rpm],10000.0,1538.77610,179.284096,1168.0,1423.0,1503.0,1612.0,2886.0
Torque [Nm],10000.0,39.98691,9.968934,3.8,33.2,40.1,46.8,76.6
Tool wear [min],10000.0,107.95100,63.654147,0.0,53.0,108.0,162.0,253.0


In [6]:
variability_df = pd.DataFrame(
    {
        "Mean": df[sensor_cols].mean(),
        "Std": df[sensor_cols].std(),
        "Variance": df[sensor_cols].var(),
    }
)

variability_df

,Mean,Std,Variance
Air temperature [K],300.00493,2.000259,4.001035
Process temperature [K],310.00556,1.483734,2.201467
Rotational speed [rpm],1538.77610,179.284096,32142.787047
Torque [Nm],39.98691,9.968934,99.379640
Tool wear [min],107.95100,63.654147,4051.850384


In [7]:
telemetry_corr = df[sensor_cols].corr()

telemetry_corr

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
Air temperature [K],1.000000,0.876107,0.022670,-0.013778,0.013853
Process temperature [K],0.876107,1.000000,0.019277,-0.014061,0.013488
Rotational speed [rpm],0.022670,0.019277,1.000000,-0.875027,0.000223
Torque [Nm],-0.013778,-0.014061,-0.875027,1.000000,-0.003093
Tool wear [min],0.013853,0.013488,0.000223,-0.003093,1.000000


In [8]:
telemetry_df = df.copy()

print(telemetry_df.shape)

(10000, 14)


In [9]:
telemetry_df.to_csv("../data/processed/telemetry_base.csv", index=False)

print("Telemetry dataset saved.")

Telemetry dataset saved.


### Telemetry Preparation Findings

- Five primary sensor signals were identified.
- Sensor variability differs significantly across measurements.
- Rotational speed exhibits the largest variance.
- These signals will be transformed into rolling, lag, and trend features in subsequent stages.

## Rolling Window Feature Engineering

Rolling statistics capture short-term operational trends and variability in machine telemetry.

Features generated:

- Rolling Mean
- Rolling Standard Deviation
- Rolling Variance

Window Size: 10 observations

In [10]:
telemetry_df = pd.read_csv("../data/processed/telemetry_base.csv")

print(telemetry_df.shape)

(10000, 14)


In [11]:
for col in sensor_cols:

    telemetry_df[f"{col}_roll_mean"] = telemetry_df[col].rolling(window=10).mean()

In [12]:
for col in sensor_cols:

    telemetry_df[f"{col}_roll_std"] = telemetry_df[col].rolling(window=10).std()

In [13]:
for col in sensor_cols:

    telemetry_df[f"{col}_roll_var"] = telemetry_df[col].rolling(window=10).var()

In [14]:
print(telemetry_df.shape)

(10000, 29)


In [15]:
telemetry_df.isnull().sum().sort_values(ascending=False).head(20)

Torque [Nm]_roll_var                 9
Air temperature [K]_roll_var         9
Tool wear [min]_roll_std             9
Torque [Nm]_roll_std                 9
Process temperature [K]_roll_var     9
Rotational speed [rpm]_roll_std      9
Process temperature [K]_roll_std     9
Tool wear [min]_roll_mean            9
Air temperature [K]_roll_std         9
Torque [Nm]_roll_mean                9
Rotational speed [rpm]_roll_mean     9
Process temperature [K]_roll_mean    9
Air temperature [K]_roll_mean        9
Tool wear [min]_roll_var             9
Rotational speed [rpm]_roll_var      9
OSF                                  0
TWF                                  0
Machine failure                      0
PWF                                  0
HDF                                  0
dtype: int64

In [16]:
telemetry_df = telemetry_df.dropna()

print(telemetry_df.shape)

(9991, 29)


In [17]:
rolling_cols = [col for col in telemetry_df.columns if "roll" in col]

rolling_cols[:10]

['Air temperature [K]_roll_mean',
 'Process temperature [K]_roll_mean',
 'Rotational speed [rpm]_roll_mean',
 'Torque [Nm]_roll_mean',
 'Tool wear [min]_roll_mean',
 'Air temperature [K]_roll_std',
 'Process temperature [K]_roll_std',
 'Rotational speed [rpm]_roll_std',
 'Torque [Nm]_roll_std',
 'Tool wear [min]_roll_std']

In [18]:
telemetry_df.to_csv("../data/processed/telemetry_rolling_features.csv", index=False)

print("Rolling feature dataset saved.")

Rolling feature dataset saved.


### Rolling Feature Findings

Rolling statistics provide localized summaries of machine behavior.

These features capture:

- Short-term trends
- Operational variability
- Signal stability

and are expected to improve predictive maintenance performance compared to raw telemetry alone.